# 1. Create a database connection 🔌🏦

In [88]:
import pandas as pd
from sqlalchemy import create_engine, types
from sqlalchemy import text # to be able to pass string

In [89]:
# 1. Set up connection (update credentials and host!)
from dotenv import dotenv_values

config = dotenv_values()

# define variables for the login
pg_user = config['POSTGRES_USER']  # align the key label with your .env file !
pg_host = config['POSTGRES_HOST']
pg_port = config['POSTGRES_PORT']
pg_db = config['POSTGRES_DB']
pg_schema = config['POSTGRES_SCHEMA']
pg_pass = config['POSTGRES_PASS']

url = f'postgresql://{pg_user}:{pg_pass}@{pg_host}:{pg_port}/{pg_db}'

engine = create_engine(url, echo=False)



In [90]:
my_schema = "team_3"


mart_customers = pd.read_sql(f"SELECT * FROM {my_schema}.mart_customers;", engine)
mart_discount_coupon = pd.read_sql(f"SELECT * FROM {my_schema}.mart_discount_coupon;", engine)
mart_holidays_2019_us = pd.read_sql(f"SELECT * FROM {my_schema}.mart_holidays_2019_us;", engine)
mart_marketing_spend = pd.read_sql(f"SELECT * FROM {my_schema}.mart_marketing_spend;", engine)
mart_online_sales = pd.read_sql(f"SELECT * FROM {my_schema}.mart_online_sales;", engine)
mart_tax_amount = pd.read_sql(f"SELECT * FROM {my_schema}.mart_tax_amount;", engine)
mart_all_data = pd.read_sql(f"SELECT * FROM {my_schema}.mart_all_data;", engine)


1. CAC (Customer Acquisition Cost)

   CAC = Total Marketing Spend in Period / Number of New Customers Acquired in Same Period

In [91]:
print(mart_customers.columns)
print(mart_customers.head())

Index(['customer_id', 'gender', 'location', 'tenure_months'], dtype='object')
   customer_id gender    location  tenure_months
0        17850      M    Illinois             12
1        13047      M  California             43
2        12583      M    Illinois             33
3        13748      F  California             30
4        15100      M  California             49


In [92]:
print(mart_online_sales.columns)
print(mart_online_sales.head())

Index(['customer_id', 'transaction_id', 'transaction_date', 'produkt_sku',
       'product_category', 'quantity', 'avg_price', 'delivery_charges',
       'coupon_status', 'revenue', 'transaction_month',
       'transaction_month_str', 'transaction_day_of_month',
       'transaction_day_of_week'],
      dtype='object')
   customer_id  transaction_id transaction_date     produkt_sku  \
0        17850           16679       2019-01-01  GGOENEBJ079499   
1        17850           16680       2019-01-01  GGOENEBJ079499   
2        17850           16681       2019-01-01  GGOEGFKQ020399   
3        17850           16682       2019-01-01  GGOEGAAB010516   
4        17850           16682       2019-01-01  GGOEGBJL013999   

  product_category  quantity  avg_price  delivery_charges coupon_status  \
0         Nest-USA         1     153.71               6.5          Used   
1         Nest-USA         1     153.71               6.5          Used   
2           Office         1       2.05             

In [93]:
print(mart_marketing_spend.columns)
print(mart_marketing_spend.head())

Index(['date', 'offline_spend', 'online_spend'], dtype='object')
         date  offline_spend  online_spend
0  2019-01-01           4500       2424.50
1  2019-01-02           4500       3480.36
2  2019-01-03           4500       1576.38
3  2019-01-04           4500       2928.55
4  2019-01-05           4500       4055.30


In [94]:
print(mart_all_data.columns)
print(mart_marketing_spend.head())

Index(['customer_id', 'customer_gender', 'customer_location',
       'customer_tenure_months', 'transaction_id', 'transaction_date',
       'produkt_sku', 'product_category', 'quantity', 'avg_price',
       'delivery_charges', 'revenue', 'coupon_status', 'discount_pct',
       'gst_onsale', 'offline_spend_that_day', 'online_spend_that_day',
       'transaction_year', 'transaction_month', 'transaction_month_str',
       'transaction_day_of_month', 'transaction_day_of_week'],
      dtype='object')
         date  offline_spend  online_spend
0  2019-01-01           4500       2424.50
1  2019-01-02           4500       3480.36
2  2019-01-03           4500       1576.38
3  2019-01-04           4500       2928.55
4  2019-01-05           4500       4055.30


In [95]:
# Make sure transaction_date is a datetime
mart_online_sales['transaction_date'] = pd.to_datetime(mart_online_sales['transaction_date'])

# Find each customer's first purchase month
first_orders = mart_online_sales.groupby('customer_id')['transaction_date'].min().reset_index()
first_orders['signup_month'] = first_orders['transaction_date'].dt.to_period('M')


In [96]:
#Step 2: Count New Customers Per Month

new_customers = first_orders.groupby('signup_month')['customer_id'].nunique().reset_index(name='new_customers')


In [97]:
#Step 3: Calculate Monthly Marketing Spend
# Ensure date is datetime (if not already)
mart_marketing_spend['date'] = pd.to_datetime(mart_marketing_spend['date'])
mart_marketing_spend['month'] = mart_marketing_spend['date'].dt.to_period('M')

# Calculate total marketing spend as the sum of both columns
mart_marketing_spend['total_spend'] = (
    mart_marketing_spend['offline_spend'].fillna(0) + 
    mart_marketing_spend['online_spend'].fillna(0)
)

# Group by month to get monthly total spend
monthly_spend = mart_marketing_spend.groupby('month')['total_spend'].sum().reset_index()

In [98]:
# 4. Merge and calculate CAC
cac = pd.merge(new_customers, monthly_spend, left_on='signup_month', right_on='month', how='inner')
cac['CAC'] = cac['total_spend'] / cac['new_customers']

print(cac[['signup_month', 'new_customers', 'total_spend', 'CAC']])

   signup_month  new_customers  total_spend          CAC
0       2019-01            215    154928.95   720.599767
1       2019-02             96    137107.92  1428.207500
2       2019-03            177    122250.09   690.678475
3       2019-04            163    157026.83   963.354785
4       2019-05            112    118259.64  1055.889643
5       2019-06            137    134318.14   980.424380
6       2019-07             94    120217.85  1278.913298
7       2019-08            135    142904.15  1058.549259
8       2019-09             78    135514.54  1737.365897
9       2019-10             87    151224.65  1738.214368
10      2019-11             68    161144.96  2369.778824
11      2019-12            106    198648.75  1874.044811


2. ROAS (Return on Ad Spend) 

ROAS = Total Revenue / Total Marketing Spend

In [99]:
#1. Adding column to online sales
mart_all_data['transaction_year'] = mart_all_data['transaction_year'].astype(int).astype(str)
mart_all_data['transaction_month'] = mart_all_data['transaction_month'].astype(int).astype(str).str.zfill(2)
mart_all_data['transaction_month_str'] = mart_all_data['transaction_year'] + '-' + mart_all_data['transaction_month']


In [100]:
#2. Total revenue per month
monthly_revenue = mart_all_data.groupby('transaction_month_str')['revenue'].sum().reset_index(name='total_revenue')

monthly_revenue

,transaction_month_str,total_revenue
0,2019-01,462866.90
1,2019-02,360036.40
2,2019-03,410408.03
3,2019-04,443100.16
4,2019-05,349159.59
5,2019-06,358594.96
6,2019-07,421362.00
7,2019-08,462309.94
8,2019-09,401553.82
9,2019-10,455643.16


In [101]:
#3 Merge with monthly marketing spend

mart_all_data['total_spend_that_day'] = mart_all_data['offline_spend_that_day'].fillna(0) + mart_all_data['online_spend_that_day'].fillna(0)
monthly_spend = mart_all_data.groupby('transaction_month_str')['total_spend_that_day'].sum().reset_index(name='total_spend')



In [102]:
roas = pd.merge(
    monthly_revenue,
    monthly_spend,
    on='transaction_month_str',
    how='inner'
)
roas['ROAS'] = roas['total_revenue'] / roas['total_spend']
print(roas[['transaction_month_str', 'total_revenue', 'total_spend', 'ROAS']])




   transaction_month_str  total_revenue  total_spend      ROAS
0                2019-01      462866.90  20052775.17  0.023082
1                2019-02      360036.40  15841536.05  0.022727
2                2019-03      410408.03  17453780.31  0.023514
3                2019-04      443100.16  21655922.13  0.020461
4                2019-05      349159.59  17525521.02  0.019923
5                2019-06      358594.96  18625403.73  0.019253
6                2019-07      421362.00  20618934.41  0.020436
7                2019-08      462309.94  28385733.77  0.016287
8                2019-09      401553.82  19257626.34  0.020852
9                2019-10      455643.16  20536272.39  0.022187
10               2019-11      541254.55  21096299.69  0.025656
11               2019-12      561140.18  28964402.01  0.019373


In [103]:
roas.head()


,transaction_month_str,total_revenue,total_spend,ROAS
0,2019-01,462866.90,20052775.17,0.023082
1,2019-02,360036.40,15841536.05,0.022727
2,2019-03,410408.03,17453780.31,0.023514
3,2019-04,443100.16,21655922.13,0.020461
4,2019-05,349159.59,17525521.02,0.019923


3. Retention rate

In [105]:
# 1. Orders: customer_id, transaction_date
orders = mart_all_data[['customer_id', 'transaction_date']].drop_duplicates()
orders['order_month'] = pd.to_datetime(orders['transaction_date']).dt.to_period('M')  # 'YYYY-MM'

# 2. First purchase month per customer
first_orders = (
    orders.groupby('customer_id')['order_month']
    .min()
    .reset_index(name='signup_month')
)

# 3. Merge for cohort tracking
cohort_data = orders.merge(first_orders, on='customer_id', how='left')

# 4. Count unique customers per cohort and order month
cohort_pivot = (
    cohort_data
    .groupby(['signup_month', 'order_month'])['customer_id']
    .nunique()
    .reset_index()
)



In [106]:
# 5. Pivot to get cohort matrix
cohort_matrix = cohort_pivot.pivot(index='signup_month', columns='order_month', values='customer_id').fillna(0)



In [107]:
# 6. New customers per cohort
new_customers = cohort_matrix.iloc[:, 0]



In [108]:
# 7. Calculate retention rate (%)
retention_rate = (
    cohort_matrix.divide(new_customers.replace(0, pd.NA), axis=0) * 100
).fillna(0).replace([float('inf'), -float('inf')], 0)

# 8. (Optional) Filter to cohorts with >0 new customers
retention_rate = retention_rate.loc[new_customers > 0]

print(retention_rate)

order_month   2019-01   2019-02    2019-03    2019-04    2019-05    2019-06  \
signup_month                                                                  
2019-01         100.0  6.046512  11.162791  15.813953  10.697674  20.465116   

order_month    2019-07    2019-08    2019-09    2019-10   2019-11    2019-12  
signup_month                                                                  
2019-01       16.27907  21.860465  10.697674  13.023256  9.302326  15.813953  


/var/folders/30/dsmll_zd0y99rxph1kl1nzc80000gp/T/ipykernel_20523/852977266.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ).fillna(0).replace([float('inf'), -float('inf')], 0)


4. AOV (AVerage order value)

AOV= Total Revenue/Number of Orders


In [126]:
monthly_orders = mart_all_data.groupby('transaction_month_str')['transaction_id'].nunique().reset_index(name='order_count')
monthly_revenue = mart_all_data.groupby('transaction_month_str')['revenue'].sum().reset_index(name='total_revenue')

monthly_aov = pd.merge(monthly_revenue, monthly_orders, on='transaction_month_str', how='inner')
monthly_aov['AOV'] = monthly_aov['total_revenue'] / monthly_aov['order_count']
print(monthly_aov[['transaction_month_str', 'AOV']])

   transaction_month_str         AOV
0                2019-01  220.203092
1                2019-02  216.368029
2                2019-03  206.131607
3                2019-04  244.401633
4                2019-05  171.661549
5                2019-06  184.842763
6                2019-07  202.577885
7                2019-08  191.511988
8                2019-09  207.843592
9                2019-10  214.420311
10               2019-11  237.184290
11               2019-12  209.068621


5. Repeat Purchase Rate

Repeat Purchase Rate= Number of Customers with >1 Order/ Total Number of Customers


In [114]:
repeat_customers = mart_all_data.groupby('customer_id')['transaction_id'].nunique().reset_index()
repeat_customers['is_repeat'] = repeat_customers['transaction_id'] > 1
repeat_rate = repeat_customers['is_repeat'].mean()
print(f"Repeat purchase rate: {repeat_rate:.2%}")

Repeat purchase rate: 91.49%


6. Churn rate

Churn Rate= Number of Customers with No Purchases for >6 Months/ Total Number of Customers

Approximation: 
Share of customers who haven’t purchased for at least 6 months, as a proxy for churn.

In [125]:
# Find last purchase date per customer
last_purchase = mart_all_data.groupby('customer_id')['transaction_date'].max().reset_index()
last_purchase['transaction_date'] = pd.to_datetime(last_purchase['transaction_date'])

reference_date = pd.to_datetime('2019-12-31')

# Calculate months since last purchase
last_purchase['months_since'] = ((reference_date - last_purchase['transaction_date']) / pd.Timedelta(days=30)).astype(int)

# Customers with >6 months inactivity are considered churned
churned = last_purchase[last_purchase['months_since'] > 6]

# Churn rate = churned customers / total unique customers
churn_rate = len(churned) / mart_all_data['customer_id'].nunique()

print(f"Churn rate (>6 months no purchase): {churn_rate:.2%}")

Churn rate (>6 months no purchase): 26.77%


7. LTV:CAC Ratio 

LTV = Total Revenue in period / Total Number of Customers acquired in period
CAC = Total Marketing Spend in period / Number of New Customers in period
LTV:CAC Ratio = LTV / CAC

In [116]:
ltv = mart_all_data.groupby('customer_id')['revenue'].sum().mean()  # Average revenue per customer

# Use your previously calculated CAC (from the 'cac' DataFrame)
avg_cac = cac['CAC'].mean()  # Average CAC, or use latest_cac = cac['CAC'].iloc[-1] for most recent

ltv_cac_ratio = ltv / avg_cac
print(f"LTV:CAC Ratio: {ltv_cac_ratio:.2f}")

LTV:CAC Ratio: 2.69


Benchmarks:

LTV:CAC > 1 → Your business is sustainable; each customer brings in more revenue than their acquisition cost.

LTV:CAC < 1 → You are overspending on customer acquisition.

8. ROAS online only

In [127]:
mart_all_data['online_spend_that_day'] = mart_all_data['online_spend_that_day'].fillna(0)
monthly_online_spend = mart_all_data.groupby('transaction_month_str')['online_spend_that_day'].sum().reset_index(name='online_spend')

In [128]:
roas_online = pd.merge(
    monthly_revenue,
    monthly_online_spend,
    on='transaction_month_str',
    how='inner'
)
roas_online['ROAS_online'] = roas_online['total_revenue'] / roas_online['online_spend']
print(roas_online[['transaction_month_str', 'total_revenue', 'online_spend', 'ROAS_online']])


   transaction_month_str  total_revenue  online_spend  ROAS_online
0                2019-01      462866.90    7686875.17     0.060215
1                2019-02      360036.40    6360136.05     0.056608
2                2019-03      410408.03    6965280.31     0.058922
3                2019-04      443100.16    8196922.13     0.054057
4                2019-05      349159.59    7574021.02     0.046100
5                2019-06      358594.96    7436403.73     0.048222
6                2019-07      421362.00    9055934.41     0.046529
7                2019-08      462309.94   11551233.77     0.040023
8                2019-09      401553.82    7371126.34     0.054477
9                2019-10      455643.16    7813772.39     0.058313
10               2019-11      541254.55    8934799.69     0.060578
11               2019-12      561140.18   11244402.01     0.049904
